# Week 2: Estimation (Individual)

In Week 1, you explored the system and developed a shared simulation framework.  
In Week 2, you will begin working individually to extract useful structure from the observed data.

## Objective

The internal state of the system is not directly observable. Your task is to design an approach that uses the available observations (and inputs) to construct a useful representation of the system.

There is **no single correct method**. You are expected to logically propose, clearly justify, and meaningfully evaluate your own approach.

## What does “estimation” mean here?

Depending on your design choices, your method need to aim to:

- identify patterns or structure in the observations,
- predict future observations,
- reconstruct hidden variables,
- compress the data into a lower-dimensional representation,
- extract features that can later be used for control.

## Tasks

You should:

1. **Define your objective**  
   Decide what you are trying to estimate and why it is useful.

2. **Design an approach**  
   Propose one or more methods based on your understanding of the system. Where possible, explore different approaches rather than relying on a single method, and compare their performance. Select the approach you consider most effective and justify your choice. Clearly state any assumptions you make.

3. **Implement your method**  
   Build on the Week 1 codebase. Test it on at least one simulation from Week 1. Your implementation does not need to be perfect, but it should be coherent and testable.

4. **Evaluate performance**  
   Use plots and quantitative measures to assess your method. Consider:
   - sensitivity to noise,
   - stability over time,
   - generalisation across different input patterns.

5. **Reflect on limitations**  
   Identify what your method does not capture and what could be improved.

## Guidance
- You may use both observations \(y(t)\) and inputs \(u(t)\). But inputs are not 100% reliable.
- You may use past data (history) if helpful.
- Simpler methods are acceptable if they are clearly justified and well analysed.
- A partially successful but well-explained approach is better than a complex but poorly understood one.

## Week 2 deliverable
**Deliverable:** Interim Report + code.
**Marks:** 20 individual marks.
**Due:** Friday 29 May at 9:00am BST. 

## Testing and evaluating your interface
On **Friday 29 May between 11am and 1pm**, the demonstrator will test your estimator to ensure that your interface meets the required specifications.  

This session is **compulsory** and important for your understanding and progression to the next stage of the project.

## Interim report guidance
Your interim report should be concise and focused, approximately **4 pages** in length, and must not exceed **5 pages total**, including figures and any appendix material.

The report should include:

- A brief description of the system and the simulations you have run,
- A summary of your Week 1 exploration,
- A clear description of your estimation approach,
- Initial results, with figures, demonstrating depth of analysis rather than broad but superficial coverage,
- A discussion of limitations and planned next steps.

The goal of the interim report is to demonstrate your understanding so far and to receive feedback ahead of the final stage.

## Connection to later work
Your estimation approach will form the basis for your control strategy in Week 3.  
Think ahead about how your representation could be used to influence the system.

## Estimation interface

To make testing and evaluation consistent across students, all estimation methods must follow the interface below.

You are free to implement any estimation strategy internally, provided that your function:
- accepts observations as input,
- returns estimated latent states and estimated inputs,
- and preserves the required function signature.

### Important:
- Do not change the function signature of `estimate_latent_and_input`. The demonstrator will call this function to evaluate your solution during the Week 2 evaluation session.
- You may implement any logic inside the function, but it must return:
   1. estimated latent states with shape (Timepoints, LatentDim)
   2. estimated inputs with shape (Timepoints, InputDim)
- If your method uses additional hyperparameters, use partial functions or wrapper functions to fix them before submission. You will not have the opportunity to adjust hyperparameters during the test. Automatic parameter tuning is welcome.


In [ ]:
from typing import Tuple
import numpy as np


def estimate_latent_and_input(
    observation: np.ndarray, LatentDim: int, InputDim: int
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Estimate the latent states and inputs from the observed neural activity.

    Parameters:
    - observation: A 2D array of shape (Timepoints, Neurons) representing the observed neural activity.
    - LatentDim: The dimensionality of the latent state space.
    - InputDim: The dimensionality of the input space.

    Returns:
    - A tuple containing:
        - latent_states: A 2D array of shape (Timepoints, LatentDim) representing the estimated latent states over time.
        - inputs: A 2D array of shape (Timepoints, InputDim) representing the estimated inputs over time.
    """
    # Placeholder implementation. Replace with actual estimation logic.
    Timepoints = observation.shape[0]
    latent_states = np.random.rand(Timepoints, LatentDim)
    inputs = np.random.rand(Timepoints, InputDim)
    

    return latent_states, inputs


# Important:
# Do not change the function signature of `estimate_latent_and_input`.
# The demonstrator will call this function to evaluate your solution.
# You may implement any logic inside the function, but it must return:
#   1. estimated latent states with shape (Timepoints, LatentDim)
#   2. estimated inputs with shape (Timepoints, InputDim)

In [ ]:
"""Single-trial blind estimation of latent states and inputs for a
linear-Gaussian state-space model.

Interface (do not change):
    estimate_latent_and_input(observation, LatentDim, InputDim)
        observation : (T, p) array of observations for ONE trial
        returns     : (x_hat, u_hat) with shapes (T, LatentDim), (T, InputDim)

Method
------
The plant is identified from the single supplied trial and the latent
state and (unobserved) input are recovered, in one call:

    x_{t+1} = A x_t + B u_t + w_t,   w ~ N(0, Q)
    y_t     = C x_t + D u_t + v_t,   v ~ N(0, R)

The unknown input is folded into an augmented state z = [x; u] with an
assumed random-walk prior u_{t+1} = u_t + eta, eta ~ N(0, Sigma_u). The
augmented system

    A_aug = [[A, B], [0, I]],  C_aug = [C, D],
    Q_aug = blkdiag(Q, Sigma_u),  R_aug = R

is an unsupervised LG-SSM, fitted by constrained EM (Kalman/RTS E-step,
closed-form M-step that updates only A, B, C, D, Q, R and holds the input
block fixed). Identification is initialised from PCA and repeated from a few
restarts; the highest-likelihood fit is smoothed to return (x_hat, u_hat).

Notes on what is and is not identifiable (see report Section 5.3):
  * the augmented system is identified only up to a similarity transform;
  * the state/input split rests on the exogeneity constraint (zero lower-left
    block) and the frozen prior (F = I, Sigma_u) -- it is assumed, not measured;
  * a residual block-upper-triangular gauge remains, so x and u are recovered
    up to invertible linear maps M_x, M_u (evaluate with aligned R^2).

All hyperparameters are frozen in code (the constants below); nothing is tuned
at evaluation time. Sigma_u is set deterministically from the supplied data.
"""

from typing import Tuple

import numpy as np

# ---- frozen hyperparameters ----------------------------------------------- #
_N_ITER = 100          # EM iterations per restart
_N_RESTARTS = 4        # random restarts; best marginal likelihood is kept
_LL_TOL = 1e-4         # relative log-likelihood convergence tolerance
_JITTER = 1e-6         # covariance regularisation


# --------------------------------------------------------------------------- #
#  Linear-Gaussian filtering / smoothing (no external input: the input is      #
#  part of the augmented state)                                                #
# --------------------------------------------------------------------------- #
def _kalman_filter(Y, A, C, Q, R, z0, P0):
    T, p = Y.shape
    na = A.shape[0]
    xf = np.zeros((T, na)); Pf = np.zeros((T, na, na))
    xp = np.zeros((T, na)); Pp = np.zeros((T, na, na))
    I = np.eye(na)
    x_prev, P_prev = z0.copy(), P0.copy()
    ll = 0.0
    for t in range(T):
        if t == 0:
            x_pred, P_pred = z0.copy(), P0.copy()
        else:
            x_pred = A @ x_prev
            P_pred = A @ P_prev @ A.T + Q
        xp[t], Pp[t] = x_pred, P_pred
        S = C @ P_pred @ C.T + R
        e = Y[t] - C @ x_pred
        # gain and update via solves (avoid explicit inverse)
        K = np.linalg.solve(S, C @ P_pred).T
        x_filt = x_pred + K @ e
        P_filt = (I - K @ C) @ P_pred
        P_filt = 0.5 * (P_filt + P_filt.T)
        xf[t], Pf[t] = x_filt, P_filt
        x_prev, P_prev = x_filt, P_filt
        sign, logdet = np.linalg.slogdet(S)
        ll += -0.5 * (logdet + e @ np.linalg.solve(S, e) + p * np.log(2 * np.pi))
    return xf, Pf, xp, Pp, ll


def _rts_smoother(A, xf, Pf, xp, Pp):
    T, na = xf.shape
    xs = xf.copy(); Ps = Pf.copy()
    Plag = np.zeros((T, na, na))
    for t in range(T - 2, -1, -1):
        J = np.linalg.solve(Pp[t + 1], A @ Pf[t]).T
        xs[t] = xf[t] + J @ (xs[t + 1] - xp[t + 1])
        Ps[t] = Pf[t] + J @ (Ps[t + 1] - Pp[t + 1]) @ J.T
        Plag[t + 1] = J @ Ps[t + 1]
    return xs, Ps, Plag


# --------------------------------------------------------------------------- #
#  Augmented-system assembly                                                   #
# --------------------------------------------------------------------------- #
def _assemble(A, B, C, D, Q, R, F, Sigma_u):
    n, m = A.shape[0], B.shape[1]
    A_aug = np.block([[A, B], [np.zeros((m, n)), F]])
    C_aug = np.hstack([C, D])
    Q_aug = np.block([[Q, np.zeros((n, m))], [np.zeros((m, n)), Sigma_u]])
    return A_aug, C_aug, Q_aug, R


# --------------------------------------------------------------------------- #
#  Constrained blind EM on a single trial                                      #
# --------------------------------------------------------------------------- #
def _fit_once(Y, n, m, F, Sigma_u, seed):
    T, p = Y.shape
    na = n + m
    rng = np.random.default_rng(seed)

    # initialise free blocks: C from PCA, A from PCA-score dynamics, B/D small
    mu = Y.mean(0, keepdims=True)
    _, _, Vt = np.linalg.svd(Y - mu, full_matrices=False)
    C = Vt[:n].T
    D = 0.1 * rng.standard_normal((p, m))
    Xpx = (Y - mu) @ C
    A = np.linalg.lstsq(Xpx[:-1], Xpx[1:], rcond=None)[0].T
    B = 0.1 * rng.standard_normal((n, m))
    Q = 0.1 * np.eye(n); R = 0.1 * np.eye(p)

    z0 = np.zeros(na)
    P0 = np.eye(na); P0[n:, n:] = Sigma_u * 10.0
    prev = -np.inf
    for _ in range(_N_ITER):
        A_aug, C_aug, Q_aug, R_aug = _assemble(A, B, C, D, Q, R, F, Sigma_u)
        xf, Pf, xp, Pp, ll = _kalman_filter(Y, A_aug, C_aug, Q_aug, R_aug, z0, P0)
        xs, Ps, Plag = _rts_smoother(A_aug, xf, Pf, xp, Pp)

        # sufficient statistics
        Szz = np.zeros((na, na)); Syz = np.zeros((p, na)); Syy = np.zeros((p, p))
        S11 = np.zeros((na, na)); S00 = np.zeros((na, na)); S10 = np.zeros((na, na))
        for t in range(T):
            Ezz = Ps[t] + np.outer(xs[t], xs[t])
            Szz += Ezz
            Syz += np.outer(Y[t], xs[t])
            Syy += np.outer(Y[t], Y[t])
            if t >= 1:
                S11 += Ezz
                S00 += Ps[t - 1] + np.outer(xs[t - 1], xs[t - 1])
                S10 += Plag[t] + np.outer(xs[t], xs[t - 1])
        Ndyn = T - 1

        # M-step: update free blocks only (input block [0 F], Sigma_u fixed)
        AB = np.linalg.solve(S00, S10.T).T
        A, B = AB[:n, :n], AB[:n, n:]
        CD = np.linalg.solve(Szz, Syz.T).T
        C, D = CD[:, :n], CD[:, n:]
        AB_top = np.hstack([A, B])
        Q = (S11[:n, :n] - AB_top @ S10[:n].T) / Ndyn
        Q = 0.5 * (Q + Q.T) + _JITTER * np.eye(n)
        R = (Syy - CD @ Syz.T) / T
        R = 0.5 * (R + R.T) + _JITTER * np.eye(p)

        if abs(ll - prev) < _LL_TOL * max(1.0, abs(prev)):
            break
        prev = ll
    return (A, B, C, D, Q, R), prev


# --------------------------------------------------------------------------- #
#  Required interface                                                          #
# --------------------------------------------------------------------------- #
def estimate_latent_and_input(
    observation: np.ndarray, LatentDim: int, InputDim: int
) -> Tuple[np.ndarray, np.ndarray]:
    Y = np.asarray(observation, dtype=float)
    if Y.ndim != 2:
        raise ValueError(f"expected observation of shape (T, p); got {Y.shape}")
    T, p = Y.shape
    n, m = int(LatentDim), int(InputDim)

    # frozen random-walk input prior; Sigma_u scaled deterministically from data
    F = np.eye(m)
    scale = float(np.var(np.diff(Y, axis=0))) if T > 1 else 1.0
    Sigma_u = np.eye(m) * max(scale, _JITTER)

    # identify the plant: several restarts, keep the highest-likelihood fit
    best_params, best_ll = None, -np.inf
    for s in range(_N_RESTARTS):
        params, ll = _fit_once(Y, n, m, F, Sigma_u, seed=s)
        if ll > best_ll:
            best_params, best_ll = params, ll

    # recover (x, u) by augmented smoothing under the identified plant
    A, B, C, D, Q, R = best_params
    A_aug, C_aug, Q_aug, R_aug = _assemble(A, B, C, D, Q, R, F, Sigma_u)
    z0 = np.zeros(n + m)
    P0 = np.eye(n + m); P0[n:, n:] = Sigma_u * 10.0
    xf, Pf, xp, Pp, _ = _kalman_filter(Y, A_aug, C_aug, Q_aug, R_aug, z0, P0)
    zs, _, _ = _rts_smoother(A_aug, xf, Pf, xp, Pp)

    x_hat = zs[:, :n]
    u_hat = zs[:, n:]
    assert x_hat.shape == (T, n) and u_hat.shape == (T, m)
    return x_hat, u_hat

In [ ]:
"""Single-trial blind estimation of latent states and inputs for a
linear-Gaussian state-space model, with sparse / heavy-tailed input priors
realised by reweighted Kalman smoothing.

Interface (do not change):
    estimate_latent_and_input(observation, LatentDim, InputDim)
        observation : (T, p) array of observations for ONE trial
        returns     : (x_hat, u_hat) of shapes (T, LatentDim), (T, InputDim)

Model
-----
    x_{t+1} = A x_t + B u_t + w_t,   w ~ N(0, Q)
    y_t     = C x_t + D u_t + v_t,   v ~ N(0, R)

The unobserved input is folded into an augmented state z = [x; u]. The input
prior is a Gaussian scale mixture on either the input INCREMENTS (Du_t, for
piecewise-constant / square / smooth inputs) or the input LEVELS (u_t, for
impulse trains). Conditioned on the per-timestep scales the prior is Gaussian,
so the inner solve is an ordinary Kalman/RTS smoother with a TIME-VARYING
input-prior covariance; the scales are updated in closed form between passes
(variational EM / iteratively reweighted smoothing). See report Section 5.3
for the derivation.

Two stages per call:
  1. Identify (A,B,C,D,Q,R) by constrained EM under a Gaussian random-walk
     input prior (robust for dynamics identification), several restarts.
  2. Recover (x,u) under the chosen reweighted prior with the plant fixed.

All hyperparameters are frozen in the constants below; the prior scale is set
deterministically from the supplied data. Nothing is tuned at evaluation time.
"""

from typing import Tuple

import numpy as np

# ---- frozen hyperparameters ----------------------------------------------- #
_N_ITER = 100          # EM iterations per restart (stage 1)
_N_RESTARTS = 4        # random restarts; best marginal likelihood kept
_LL_TOL = 1e-4         # EM convergence tolerance
_N_OUTER = 8           # reweighting passes (stage 2)
_T_DOF = 3.0           # Student-t degrees of freedom for the increment prior
_DEFAULT_PRIOR = "student_t_incr"   # robust default (see module docstring)
_JITTER = 1e-6


# --------------------------------------------------------------------------- #
#  Linear-Gaussian filtering / smoothing. Q may be constant (na,na) or         #
#  time-varying (T,na,na); time-varying Q[t] is the noise on the t-1 -> t      #
#  transition, which is how the per-timestep input-prior scale enters.         #
# --------------------------------------------------------------------------- #
def _kalman_filter(Y, A, C, Q, R, z0, P0):
    T, p = Y.shape
    na = A.shape[0]
    tv = (Q.ndim == 3)
    xf = np.zeros((T, na)); Pf = np.zeros((T, na, na))
    xp = np.zeros((T, na)); Pp = np.zeros((T, na, na))
    I = np.eye(na)
    x_prev, P_prev = z0.copy(), P0.copy()
    ll = 0.0
    for t in range(T):
        if t == 0:
            x_pred, P_pred = z0.copy(), P0.copy()
        else:
            Qt = Q[t] if tv else Q
            x_pred = A @ x_prev
            P_pred = A @ P_prev @ A.T + Qt
        xp[t], Pp[t] = x_pred, P_pred
        S = C @ P_pred @ C.T + R
        e = Y[t] - C @ x_pred
        K = np.linalg.solve(S, C @ P_pred).T
        x_filt = x_pred + K @ e
        P_filt = (I - K @ C) @ P_pred
        P_filt = 0.5 * (P_filt + P_filt.T)
        xf[t], Pf[t] = x_filt, P_filt
        x_prev, P_prev = x_filt, P_filt
        _, logdet = np.linalg.slogdet(S)
        ll += -0.5 * (logdet + e @ np.linalg.solve(S, e) + p * np.log(2 * np.pi))
    return xf, Pf, xp, Pp, ll


def _rts_smoother(A, xf, Pf, xp, Pp):
    T, na = xf.shape
    xs = xf.copy(); Ps = Pf.copy()
    Plag = np.zeros((T, na, na))
    for t in range(T - 2, -1, -1):
        J = np.linalg.solve(Pp[t + 1], A @ Pf[t]).T
        xs[t] = xf[t] + J @ (xs[t + 1] - xp[t + 1])
        Ps[t] = Pf[t] + J @ (Ps[t + 1] - Pp[t + 1]) @ J.T
        Plag[t + 1] = J @ Ps[t + 1]
    return xs, Ps, Plag


def _assemble(A, B, C, D, Q, R, F, Sigma_u):
    n, m = A.shape[0], B.shape[1]
    A_aug = np.block([[A, B], [np.zeros((m, n)), F]])
    C_aug = np.hstack([C, D])
    Q_aug = np.block([[Q, np.zeros((n, m))], [np.zeros((m, n)), Sigma_u]])
    return A_aug, C_aug, Q_aug, R


# --------------------------------------------------------------------------- #
#  Stage 1: constrained Gaussian EM identification (random-walk input prior)   #
# --------------------------------------------------------------------------- #
def _fit_once(Y, n, m, F, Sigma_u, seed):
    T, p = Y.shape
    na = n + m
    rng = np.random.default_rng(seed)
    mu = Y.mean(0, keepdims=True)
    _, _, Vt = np.linalg.svd(Y - mu, full_matrices=False)
    C = Vt[:n].T
    D = 0.1 * rng.standard_normal((p, m))
    Xpx = (Y - mu) @ C
    A = np.linalg.lstsq(Xpx[:-1], Xpx[1:], rcond=None)[0].T
    B = 0.1 * rng.standard_normal((n, m))
    Q = 0.1 * np.eye(n); R = 0.1 * np.eye(p)
    z0 = np.zeros(na); P0 = np.eye(na); P0[n:, n:] = Sigma_u * 10.0
    prev = -np.inf
    for _ in range(_N_ITER):
        A_aug, C_aug, Q_aug, R_aug = _assemble(A, B, C, D, Q, R, F, Sigma_u)
        xf, Pf, xp, Pp, ll = _kalman_filter(Y, A_aug, C_aug, Q_aug, R_aug, z0, P0)
        xs, Ps, Plag = _rts_smoother(A_aug, xf, Pf, xp, Pp)
        Szz = np.zeros((na, na)); Syz = np.zeros((p, na)); Syy = np.zeros((p, p))
        S11 = np.zeros((na, na)); S00 = np.zeros((na, na)); S10 = np.zeros((na, na))
        for t in range(T):
            Ezz = Ps[t] + np.outer(xs[t], xs[t])
            Szz += Ezz; Syz += np.outer(Y[t], xs[t]); Syy += np.outer(Y[t], Y[t])
            if t >= 1:
                S11 += Ezz
                S00 += Ps[t - 1] + np.outer(xs[t - 1], xs[t - 1])
                S10 += Plag[t] + np.outer(xs[t], xs[t - 1])
        Ndyn = T - 1
        AB = np.linalg.solve(S00, S10.T).T
        A, B = AB[:n, :n], AB[:n, n:]
        CD = np.linalg.solve(Szz, Syz.T).T
        C, D = CD[:, :n], CD[:, n:]
        AB_top = np.hstack([A, B])
        Q = (S11[:n, :n] - AB_top @ S10[:n].T) / Ndyn
        Q = 0.5 * (Q + Q.T) + _JITTER * np.eye(n)
        R = (Syy - CD @ Syz.T) / T
        R = 0.5 * (R + R.T) + _JITTER * np.eye(p)
        if abs(ll - prev) < _LL_TOL * max(1.0, abs(prev)):
            break
        prev = ll
    return (A, B, C, D, Q, R), prev


def _identify(Y, n, m, F, Sigma_u):
    best, best_ll = None, -np.inf
    for s in range(_N_RESTARTS):
        params, ll = _fit_once(Y, n, m, F, Sigma_u, seed=s)
        if ll > best_ll:
            best, best_ll = params, ll
    return best


# --------------------------------------------------------------------------- #
#  Stage 2: input recovery by reweighted smoothing (Gaussian scale mixture).   #
#  prior in {"gaussian", "student_t_incr", "sparse_level"}.                    #
# --------------------------------------------------------------------------- #
def _recover_input(Y, params, n, m, prior, Sigma_u_base, nu=_T_DOF,
                   n_outer=_N_OUTER):
    A, B, C, D, Q, R = params
    T, p = Y.shape
    na = n + m
    # F = I for increment priors (random walk), F = 0 for level/sparse priors
    F = np.zeros((m, m)) if prior == "sparse_level" else np.eye(m)
    A_aug, C_aug, _, R_aug = _assemble(A, B, C, D, Q, R, F, Sigma_u_base)
    z0 = np.zeros(na); P0 = np.eye(na); P0[n:, n:] = Sigma_u_base * 10.0
    scale_sq = np.maximum(np.diag(Sigma_u_base).copy(), _JITTER)   # (m,)
    Su = np.tile(Sigma_u_base, (T, 1, 1))                          # (T,m,m)
    xs = None
    for _ in range(max(1, n_outer)):
        Qarr = np.zeros((T, na, na))
        Qarr[:, :n, :n] = Q
        Qarr[:, n:, n:] = Su
        xf, Pf, xp, Pp, _ = _kalman_filter(Y, A_aug, C_aug, Qarr, R_aug, z0, P0)
        xs, Ps, Plag = _rts_smoother(A_aug, xf, Pf, xp, Pp)
        if prior == "gaussian":
            break
        u = xs[:, n:]
        Esq = np.zeros((T, m))
        Puu = np.array([np.diag(Ps[t][n:, n:]) for t in range(T)])  # (T,m)
        if prior == "student_t_incr":
            for t in range(1, T):
                cross = np.diag(Plag[t][n:, n:])
                var = np.maximum(Puu[t] + Puu[t - 1] - 2 * cross, 0.0)
                d = u[t] - u[t - 1]
                Esq[t] = d * d + var
            Esq[0] = Esq[1] if T > 1 else scale_sq
        else:  # sparse_level
            Esq = u * u + Puu
        # variational reweight: prior precision w_t = E[1/tau_t | .]
        w = (nu + 1.0) / (nu * scale_sq[None, :] + Esq)            # (T,m)
        inv = 1.0 / np.maximum(w, 1e-12)
        Su = np.zeros((T, m, m))
        for t in range(T):
            Su[t] = np.diag(inv[t])
    return xs[:, :n], xs[:, n:]


# --------------------------------------------------------------------------- #
#  Required interface                                                          #
# --------------------------------------------------------------------------- #
def estimate_latent_and_input(
    observation: np.ndarray, LatentDim: int, InputDim: int
) -> Tuple[np.ndarray, np.ndarray]:
    Y = np.asarray(observation, dtype=float)
    if Y.ndim != 2:
        raise ValueError(f"expected observation of shape (T, p); got {Y.shape}")
    T, p = Y.shape
    n, m = int(LatentDim), int(InputDim)

    scale = float(np.var(np.diff(Y, axis=0))) if T > 1 else 1.0
    Sigma_u = np.eye(m) * max(scale, _JITTER)

    # stage 1: identify the plant (Gaussian random-walk prior, restarts)
    params = _identify(Y, n, m, np.eye(m), Sigma_u)

    # stage 2: recover (x, u) under the robust reweighted increment prior
    x_hat, u_hat = _recover_input(Y, params, n, m, _DEFAULT_PRIOR, Sigma_u)

    assert x_hat.shape == (T, n) and u_hat.shape == (T, m)
    return x_hat, u_hat